In [0]:
%python
# Databricks notebook source
# COMMAND ----------
# DBTITLE 1,1. Parametrización del Entorno
# Se declara el widget que recibirá 'dev' o 'prod'
dbutils.widgets.text("environment", "dev", "Ambiente de ejecución")
env = dbutils.widgets.get("environment")

print(f"--> Ejecutando pipeline en ambiente: {env.upper()}")

bronze_db = f"{env}_bronze"
silver_db = f"{env}_silver"

# COMMAND ----------
# DBTITLE 1,2. Creación de Esquemas / Bases de Datos
spark.sql(f"CREATE DATABASE IF NOT EXISTS {bronze_db}")
spark.sql(f"CREATE DATABASE IF NOT EXISTS {silver_db}")

# COMMAND ----------
# DBTITLE 1,3. Capa Bronze (Ingesta cruda simulada)
raw_data = [
    (101, "CAMPAIGN_A", 1200.0),
    (102, "CAMPAIGN_B", 3500.0),
    (102, "CAMPAIGN_B", 3500.0),  # Duplicado intencional
    (103, None, 450.0)             # Nulo intencional
]
columns = ["campaign_id", "campaign_name", "budget"]

df_bronze = spark.createDataFrame(raw_data, columns)

df_bronze.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{bronze_db}.raw_campaigns")

print(f"Capa Bronze generada en: {bronze_db}.raw_campaigns")

# COMMAND ----------
# DBTITLE 1,4. Capa Silver (Limpieza y deduplicación)
df_bronze_read = spark.read.table(f"{bronze_db}.raw_campaigns")

# Reglas: descartar nulos en campaña y eliminar duplicados por id
df_silver = (
    df_bronze_read
    .filter("campaign_name IS NOT NULL")
    .dropDuplicates(["campaign_id"])
)

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{silver_db}.clean_campaigns")

print(f"Capa Silver generada en: {silver_db}.clean_campaigns")

In [0]:
%sh
git config user.name "FedeGKL"
git config user.email "kuhllamas98@hotmail.com"
git add -A
git commit -m "feat: definicion de bundle y workflow medallion" || true
git push -u origin main